# Notes

__Filtering rare ASVs:__
Remove ASVs with frequency lower than 0.1% of mean sample depth. Find the mean multiply by 0.001 for cutoff. Based on: 

Comeau AM, Douglas GM, Langille MGI. 2017. Microbiome Helper: a Custom and Streamlined Workflow for Microbiome Research. mSystems 2(1): e00127-16; DOI: 10.1128/mSystems.00127-16 (https://journals.asm.org/doi/10.1128/msystems.00127-16)


# Merge Years:

In [2]:
### 1. Merge ASV tables:
import os
qiimedir = "./merged_data"
os.makedirs(qiimedir, exist_ok=True)

!qiime feature-table merge \
  --i-tables \
        ../2020/2020_filtered_table.qza \
        ../2023/2023_filtered_table.qza \
  --p-overlap-method 'error_on_overlapping_sample' \
  --o-merged-table merged.qza \
  --verbose

Saved FeatureTable[Frequency] to: merged.qza


In [6]:
### 2. Summarize the merged table
!qiime feature-table summarize \
  --i-table merged.qza \
  --output-dir merged1 \
  --m-metadata-file fungiAll.txt

Saved ImmutableMetadata to: merged1/feature_frequencies.qza
Saved ImmutableMetadata to: merged1/sample_frequencies.qza
Saved Visualization to: merged1/summary.qzv


In [8]:
### 3. Merge ASV sequences:

!qiime feature-table merge-seqs \
  --i-data ../2020/2020_filtered_sequences.qza ../2023/2023_filtered_sequences.qza \
  --o-merged-data merged_seqs.qza

Saved FeatureData[Sequence] to: merged_seqs.qza


In [9]:
### 4. Cluster OTUs
!qiime vsearch cluster-features-de-novo \
  --i-sequences merged_seqs.qza \
  --i-table merged.qza \
  --p-perc-identity .97 \
  --p-threads 10 \
  --o-clustered-table merged_otus.qza \
  --o-clustered-sequences merged_otus_seqs.qza \
  --verbose

Running external command line application. This may print messages to stdout and/or stderr.
The command being run is below. This command cannot be manually re-run as it will depend on temporary files that no longer exist.

Command: vsearch --cluster_size /var/folders/86/7zzx0b615d3fp0f0x2ycc7gw0000gq/T/tmp5_y88jn2 --id 0.97 --centroids /var/folders/86/7zzx0b615d3fp0f0x2ycc7gw0000gq/T/qiime2/ac044747/processes/45973-1780430833.344796@ac044747/tmp/rachis-OutPath-b2e7qkbn --uc /var/folders/86/7zzx0b615d3fp0f0x2ycc7gw0000gq/T/tmp5fg2i7pe --strand plus --qmask none --xsize --threads 10 --minseqlength 1 --fasta_width 0

vsearch v2.30.4_macos_x86_64, 48.0GB RAM, 15 cores
https://github.com/torognes/vsearch

Reading file /var/folders/86/7zzx0b615d3fp0f0x2ycc7gw0000gq/T/tmp5_y88jn2 100%  
1752314 nt in 6864 seqs, min 222, max 423, avg 255
Sorting by abundance 100%
Counting k-mers 100% 
Clustering 100%                                          
Sorting clusters 100%
Writing clusters 100% 
Cluster

In [12]:
### 5. Summarize OTU table
!qiime feature-table summarize \
  --i-table merged_otus.qza \
  --output-dir merged2 \
  --m-metadata-file fungiAll.txt

Saved ImmutableMetadata to: merged2/feature_frequencies.qza
Saved ImmutableMetadata to: merged2/sample_frequencies.qza
Saved Visualization to: merged2/summary.qzv


In [ ]:
# ### 6. Download taxonomy database
# !qiime rescript get-unite-data \
#   --p-version '2025-02-19' \
#   --p-cluster-id 'dynamic' \
#   --o-taxonomy unite_10_taxonomy.qza \
#   --o-sequences unite_10_seqs.qza

Saved FeatureData[Taxonomy] to: unite_10_taxonomy.qza
Saved FeatureData[Sequence] to: unite_10_seqs.qza


In [2]:
### 6. Assign taxonomy
!qiime feature-classifier classify-consensus-vsearch \
        --i-query merged_otus_seqs.qza \
        --i-reference-reads unite_97.qza \
        --i-reference-taxonomy unite_97_tax.qza \
        --p-threads 10 \
        --o-classification taxonomy-vsearch.qza \
        --o-search-results taxonomy-vsearch-search.qza

Saved FeatureData[Taxonomy] to: taxonomy-vsearch.qza
Saved FeatureData[BLAST6] to: taxonomy-vsearch-search.qza


In [3]:
### 7. Filter taxonomy to remove unassigned and non-fungal taxa
!qiime taxa filter-table \
  --i-table merged_otus.qza \
  --i-taxonomy taxonomy-vsearch.qza \
  --p-include 'k__Fungi' \
  --o-filtered-table merged_otus_fungi.qza
  
!qiime feature-table summarize \
  --i-table merged_otus_fungi.qza \
  --output-dir merged3-fungi \
  --m-metadata-file fungiAll.txt

Saved FeatureTable[Frequency] to: merged_otus_fungi.qza
Saved ImmutableMetadata to: merged3-fungi/feature_frequencies.qza
Saved ImmutableMetadata to: merged3-fungi/sample_frequencies.qza
Saved Visualization to: merged3-fungi/summary.qzv


## Analyses:

### Per site

In [4]:
!qiime feature-table filter-samples \
  --i-table merged_otus_fungi.qza \
  --m-metadata-file fungiAll.txt \
  --p-where "Year=2023" \
  --o-filtered-table merged_otus_fungi_2023.qza
  
!qiime feature-table summarize \
  --i-table merged_otus_fungi_2023.qza \
  --output-dir merged3-fungi-2023 \
  --m-metadata-file fungiAll.txt


Saved FeatureTable[Frequency] to: merged_otus_fungi_2023.qza
Saved ImmutableMetadata to: merged3-fungi-2023/feature_frequencies.qza
Saved ImmutableMetadata to: merged3-fungi-2023/sample_frequencies.qza
Saved Visualization to: merged3-fungi-2023/summary.qzv


In [5]:
!qiime diversity core-metrics \
  --i-table merged_otus_fungi_2023.qza \
  --p-sampling-depth 6500 \
  --m-metadata-file fungiAll.txt \
  --p-n-jobs 10 \
  --output-dir merged_otus_fungi_2023_core-metrics

Saved FeatureTable[Frequency] to: merged_otus_fungi_2023_core-metrics/rarefied_table.qza
Saved SampleData[AlphaDiversity] to: merged_otus_fungi_2023_core-metrics/observed_features_vector.qza
Saved SampleData[AlphaDiversity] to: merged_otus_fungi_2023_core-metrics/shannon_vector.qza
Saved SampleData[AlphaDiversity] to: merged_otus_fungi_2023_core-metrics/evenness_vector.qza
Saved DistanceMatrix to: merged_otus_fungi_2023_core-metrics/jaccard_distance_matrix.qza
Saved DistanceMatrix to: merged_otus_fungi_2023_core-metrics/bray_curtis_distance_matrix.qza
Saved PCoAResults to: merged_otus_fungi_2023_core-metrics/jaccard_pcoa_results.qza
Saved PCoAResults to: merged_otus_fungi_2023_core-metrics/bray_curtis_pcoa_results.qza
Saved Visualization to: merged_otus_fungi_2023_core-metrics/jaccard_emperor.qzv
Saved Visualization to: merged_otus_fungi_2023_core-metrics/bray_curtis_emperor.qzv


In [6]:
!qiime taxa barplot \
  --i-table merged_otus_fungi_2023.qza \
  --i-taxonomy taxonomy-vsearch.qza \
  --m-metadata-file fungiAll.txt \
  --o-visualization merged_otus_fungi_2023_taxa_barplot.qzv

Saved Visualization to: merged_otus_fungi_2023_taxa_barplot.qzv


### Kaa'la per year

In [7]:
!qiime feature-table filter-samples \
  --i-table merged_otus_fungi.qza \
  --m-metadata-file fungiAll.txt \
  --p-where "Site='Kaala' AND Preference!='UndefinedP'" \
  --o-filtered-table merged_otus_fungi_Kaala.qza
  
!qiime feature-table summarize \
  --i-table merged_otus_fungi_Kaala.qza \
  --output-dir merged3-fungi-Kaala \
  --m-metadata-file fungiAll.txt


Saved FeatureTable[Frequency] to: merged_otus_fungi_Kaala.qza
Saved ImmutableMetadata to: merged3-fungi-Kaala/feature_frequencies.qza
Saved ImmutableMetadata to: merged3-fungi-Kaala/sample_frequencies.qza
Saved Visualization to: merged3-fungi-Kaala/summary.qzv


In [8]:
!qiime diversity core-metrics \
  --i-table merged_otus_fungi_Kaala.qza \
  --p-sampling-depth 5000 \
  --m-metadata-file fungiAll.txt \
  --p-n-jobs 10 \
  --output-dir merged_otus_fungi_Kaala_core-metrics

Saved FeatureTable[Frequency] to: merged_otus_fungi_Kaala_core-metrics/rarefied_table.qza
Saved SampleData[AlphaDiversity] to: merged_otus_fungi_Kaala_core-metrics/observed_features_vector.qza
Saved SampleData[AlphaDiversity] to: merged_otus_fungi_Kaala_core-metrics/shannon_vector.qza
Saved SampleData[AlphaDiversity] to: merged_otus_fungi_Kaala_core-metrics/evenness_vector.qza
Saved DistanceMatrix to: merged_otus_fungi_Kaala_core-metrics/jaccard_distance_matrix.qza
Saved DistanceMatrix to: merged_otus_fungi_Kaala_core-metrics/bray_curtis_distance_matrix.qza
Saved PCoAResults to: merged_otus_fungi_Kaala_core-metrics/jaccard_pcoa_results.qza
Saved PCoAResults to: merged_otus_fungi_Kaala_core-metrics/bray_curtis_pcoa_results.qza
Saved Visualization to: merged_otus_fungi_Kaala_core-metrics/jaccard_emperor.qzv
Saved Visualization to: merged_otus_fungi_Kaala_core-metrics/bray_curtis_emperor.qzv


### Export OTU table

In [9]:
!qiime feature-table relative-frequency \
  --i-table merged_otus_fungi_2023.qza \
  --o-relative-frequency-table merged_otus_fungi_2023_frequency.qza

!qiime tools export \
  --input-path merged_otus_fungi_2023_frequency.qza \
  --output-path exported-table

!qiime tools export \
  --input-path taxonomy-vsearch.qza \
  --output-path exported-taxonomy

Saved FeatureTable[RelativeFrequency] to: merged_otus_fungi_2023_frequency.qza
Exported merged_otus_fungi_2023_frequency.qza as BIOMV210DirFmt to directory exported-table
Exported taxonomy-vsearch.qza as TSVTaxonomyDirectoryFormat to directory exported-taxonomy


In [10]:
# Change "Feature ID" to "#OTUID" and "Taxon" to "taxonomy"
!sed 's/Feature ID/#OTUID/g; s/Taxon/taxonomy/g' \
  exported-taxonomy/taxonomy.tsv > exported-taxonomy/taxonomy-reformatted.tsv

In [11]:
!biom add-metadata \
  -i ./exported-table/feature-table.biom \
  -o ./exported-table/table-with-taxonomy.biom \
  --observation-metadata-fp ./exported-taxonomy/taxonomy-reformatted.tsv \
  --sc-separated taxonomy

In [12]:
!biom convert \
  -i ./exported-table/table-with-taxonomy.biom \
  -o ./exported-table/otu-table-with-taxonomy.tsv \
  --to-tsv \
  --header-key taxonomy

## Files:

- Bray-Curtis distance Matrix for site analysis: merged_otus_fungi_2023_core-metrics/bray_curtis_distance_matrix.qza
- Bray-Curtis distance Matrix for Interannual analysis: merged_otus_fungi_Kaala_core-metrics/bray_curtis_distance_matrix.qza
- Alpha-Diversity measures: 
  - Pielou: merged_otus_fungi_2023_core-metrics/evenness_vector.qza
  - Shannon: merged_otus_fungi_2023_core-metrics/shannon_vector.qza
  - Richness: merged_otus_fungi_2023_core-metrics/observed_features_vector.qza